# Ders 2: Değer Tahmin Problemleri
Bu bölümde, bir Markov Ödül Süreci (MRP) altındaki değer fonksiyonunu **etkileşim yoluyla** (model bilmeden) nasıl tahmin edeceğimizi öğreneceğiz.

---
**Konular:**
1. Tabular TD(0) — Temporal Difference Öğrenme
2. Every-Visit Monte Carlo
3. TD(λ) — Birleşik Yaklaşım
4. Büyük Durum Uzayları: Fonksiyon Yaklaşımı
5. Gradient TD Öğrenme
6. Least-Squares TD (LSTD)


## 2.1 Kurulum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# ── Bölüm 1'den MDP sınıfı (kısaltılmış) ──────────────────────────
class MRP:
    """Markov Ödül Süreci — değer tahmin için kullanılır."""
    def __init__(self, states, transitions, rewards, gamma=0.95):
        self.states = states
        self.n_states = len(states)
        self.state_idx = {s: i for i, s in enumerate(states)}
        self.transitions = transitions   # {s: {s': prob}}
        self.rewards = rewards           # {s: float}
        self.gamma = gamma
    
    def step(self, state):
        """Bir geçiş simüle eder → (next_state, reward)"""
        next_states = list(self.transitions[state].keys())
        probs       = list(self.transitions[state].values())
        next_state  = np.random.choice(next_states, p=probs)
        reward      = self.rewards.get(state, 0.0)
        return next_state, reward
    
    def true_value(self):
        """Bellman doğrusal sistemini doğrudan çözer: V = r + γPV"""
        n  = self.n_states
        R  = np.array([self.rewards.get(s, 0.0) for s in self.states])
        P  = np.zeros((n, n))
        for i, s in enumerate(self.states):
            for s_next, prob in self.transitions.get(s, {}).items():
                j = self.state_idx[s_next]
                P[i, j] = prob
        # V = (I - γP)^{-1} r
        V = np.linalg.solve(np.eye(n) - self.gamma * P, R)
        return V

def create_random_walk(n_states=7, gamma=0.99):
    """
    Klasik 'Random Walk' MRP (Sutton & Barto'dan uyarlandı).
    n_states durum, uçlarda terminal.
    Sağa gidince +1 ödül, sola gidince -1 ödül.
    """
    states      = list(range(n_states))
    transitions = {}
    rewards     = {}
    for s in states:
        if s == 0 or s == n_states - 1:
            transitions[s] = {s: 1.0}
            rewards[s]     = 0.0
        else:
            transitions[s] = {s - 1: 0.5, s + 1: 0.5}
            rewards[s]     = 1.0 if s == n_states - 2 else 0.0
    return MRP(states, transitions, rewards, gamma)

mrp = create_random_walk(n_states=7)
V_true = mrp.true_value()
print("Random Walk MRP oluşturuldu.")
print(f"Gerçek değer fonksiyonu V*: {np.round(V_true, 3)}")

## 2.2 Tabular TD(0) Algoritması

### Algoritma 1 (Szepesvári, s.12)

```
function TD0(X, R, Y, V):
    δ ← R + γ·V[Y] − V[X]
    V[X] ← V[X] + α·δ
    return V
```

**TD hatası (temporal difference error):**
$$\delta_{t+1} = R_{t+1} + \gamma \hat{V}_t(X_{t+1}) - \hat{V}_t(X_t)$$

**Güncelleme:**
$$\hat{V}_{t+1}(x) = \hat{V}_t(x) + \alpha_t \, \delta_{t+1} \, \mathbf{1}[X_t = x]$$

**Bootstrapping:** TD(0), hedef olarak bir sonraki adımın tahminini kullanır — bu onu Monte Carlo'dan ayıran temel özelliktir.

**Yakınsama garantisi:** Robbins-Monro koşulları sağlandığında ($\sum \alpha_t = \infty$, $\sum \alpha_t^2 < \infty$), $\hat{V}_t \to V$ neredeyse kesin olarak.


In [ ]:
def td0(mrp, n_steps=50_000, alpha=0.05, alpha_decay=False):
    """
    Tabular TD(0) — Algoritma 1 (Szepesvári)
    
    Adım büyüklüğü:
      sabit  : α_t = alpha
      azalan : α_t = alpha / (1 + t^0.6)  [Robbins-Monro koşulunu sağlar]
    """
    V      = np.zeros(mrp.n_states)
    errors = []
    state  = mrp.states[len(mrp.states) // 2]  # Orta durumdan başla
    
    for t in range(n_steps):
        next_state, reward = mrp.step(state)
        
        # Adım büyüklüğü
        if alpha_decay:
            alpha_t = alpha / (1 + t ** 0.6)
        else:
            alpha_t = alpha
        
        # TD(0) güncellemesi
        i = mrp.state_idx[state]
        j = mrp.state_idx[next_state]
        td_error = reward + mrp.gamma * V[j] - V[i]
        V[i]    += alpha_t * td_error
        
        # RMSE kaydı (her 500 adımda)
        if t % 500 == 0:
            rmse = np.sqrt(np.mean((V - V_true) ** 2))
            errors.append(rmse)
        
        # Terminal durumda sıfırla
        if next_state in [0, mrp.n_states - 1]:
            state = mrp.states[len(mrp.states) // 2]
        else:
            state = next_state
    
    return V, errors

# Farklı α değerleri dene
alphas = [0.01, 0.05, 0.1, 0.3]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

all_V  = {}
all_err = {}
for alpha in alphas:
    V_est, errs = td0(mrp, n_steps=30_000, alpha=alpha)
    all_V[alpha]   = V_est
    all_err[alpha] = errs

# Sol: Tahmin edilen değer fonksiyonları
x_labels = [f's{i}' for i in mrp.states]
x_pos    = np.arange(mrp.n_states)
axes[0].plot(x_pos, V_true, 'k--', linewidth=3, label='Gerçek V', zorder=5)
for alpha in alphas:
    axes[0].plot(x_pos, all_V[alpha], '-o', markersize=6, linewidth=2, label=f'α={alpha}')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(x_labels)
axes[0].set_xlabel('Durum')
axes[0].set_ylabel('Değer Tahmini')
axes[0].set_title('TD(0): Farklı Öğrenme Hızları (30K adım)', fontweight='bold')
axes[0].legend()

# Sağ: Yakınsama eğrileri
step_ticks = np.arange(len(all_err[0.05])) * 500
for alpha in alphas:
    axes[1].plot(step_ticks, all_err[alpha], linewidth=2, label=f'α={alpha}')
axes[1].set_xlabel('Adım Sayısı')
axes[1].set_ylabel('RMSE')
axes[1].set_title('TD(0) Yakınsama Eğrisi (RMSE)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_td0.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Every-Visit Monte Carlo

### Algoritma 2 (Szepesvári, s.15)

Episodik bir MRP'de, her epizodun sonunda geri dönerek getirileri hesapla:

```
function EveryVisitMC(X₀, R₁, X₁, ..., X_{T-1}, R_T, V):
    sum ← 0
    for t = T-1 downto 0:
        sum ← R_{t+1} + γ·sum         ← geriye doğru toplam (verimli!)
        V[Xₜ] ← V[Xₜ] + α·(sum - V[Xₜ])
    return V
```

**Getiri:** $R_t = \sum_{s=t}^{T-1} \gamma^{s-t} R_{s+1}$ (çok-adımlı tahmin)

**Fark:** Monte Carlo önyükleme **yapmaz** — tam getirileri kullanır.


In [ ]:
def generate_episode(mrp, start_state=None, max_steps=500):
    """Bir epizot üretir: (durum, ödül) dizisi"""
    if start_state is None:
        start_state = mrp.states[len(mrp.states) // 2]
    
    states_ep  = [start_state]
    rewards_ep = []
    state      = start_state
    
    for _ in range(max_steps):
        next_state, reward = mrp.step(state)
        rewards_ep.append(reward)
        states_ep.append(next_state)
        state = next_state
        if next_state in [0, mrp.n_states - 1]:  # Terminal
            break
    
    return states_ep[:-1], rewards_ep

def every_visit_mc(mrp, n_episodes=5_000, alpha=0.05):
    """
    Every-Visit Monte Carlo — Algoritma 2 (Szepesvári)
    """
    V      = np.zeros(mrp.n_states)
    errors = []
    
    for ep in range(n_episodes):
        states_ep, rewards_ep = generate_episode(mrp)
        T = len(rewards_ep)
        
        G = 0.0  # Geriye doğru kümülatif getiri
        for t in range(T - 1, -1, -1):
            G     = rewards_ep[t] + mrp.gamma * G
            s     = states_ep[t]
            i     = mrp.state_idx[s]
            V[i] += alpha * (G - V[i])
        
        if ep % 50 == 0:
            rmse = np.sqrt(np.mean((V - V_true) ** 2))
            errors.append(rmse)
    
    return V, errors

# TD(0) vs Monte Carlo karşılaştırması
V_mc,  err_mc  = every_visit_mc(mrp, n_episodes=5_000, alpha=0.05)
V_td,  err_td  = td0(mrp, n_steps=50_000, alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sol: Değer karşılaştırması
x_pos = np.arange(mrp.n_states)
width = 0.25
axes[0].bar(x_pos - width, V_true, width, label='Gerçek V',    color='black',  alpha=0.7)
axes[0].bar(x_pos,          V_td,  width, label='TD(0)',        color='steelblue', alpha=0.8)
axes[0].bar(x_pos + width,  V_mc,  width, label='Monte Carlo',  color='darkorange', alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f's{i}' for i in mrp.states])
axes[0].set_xlabel('Durum')
axes[0].set_ylabel('Değer')
axes[0].set_title('TD(0) vs Monte Carlo: Değer Tahminleri', fontweight='bold')
axes[0].legend()

# Sağ: Yakınsama (epizot/adım bazında)
ep_ticks = np.arange(len(err_mc)) * 50
step_ticks_td = np.arange(len(err_td)) * 500

ax2 = axes[1]
ax2.plot(ep_ticks,       err_mc, 'darkorange', linewidth=2, label=f'MC (α=0.05, RMSE={err_mc[-1]:.4f})')
# TD için adımı epizoda çevir (yaklaşık)
ax2.plot(np.linspace(0, 5000, len(err_td)), err_td, 'steelblue', linewidth=2,
         label=f'TD(0) (α=0.05, RMSE={err_td[-1]:.4f})')
ax2.set_xlabel('Epizot (veya adım eşdeğeri)')
ax2.set_ylabel('RMSE')
ax2.set_title('Yakınsama Karşılaştırması', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_mc_vs_td.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Final RMSE → MC: {err_mc[-1]:.5f}  |  TD(0): {err_td[-1]:.5f}")

## 2.4 TD(λ) — Birleşik Yaklaşım

TD(λ), TD(0) ile Monte Carlo'yu **eligibility trace** (uygunluk izi) mekanizmasıyla birleştirir.

### n-adım Getiri
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \hat{V}(X_{t+n})$$

- $n=1$ → TD(0)  
- $n=\infty$ → Monte Carlo

### λ-getirisi (geometrik ağırlıklı ortalama)
$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

### Eligibility Trace (Uygunluk İzi) Güncellemesi

$$z_{t+1}(x) = \gamma \lambda \, z_t(x) + \mathbf{1}[X_t = x]$$
$$\hat{V}_{t+1}(x) = \hat{V}_t(x) + \alpha_t \, \delta_{t+1} \, z_{t+1}(x)$$


In [ ]:
def td_lambda(mrp, lam=0.5, alpha=0.05, n_episodes=3_000):
    """
    TD(λ) — Uygunluk izleriyle geriye dönük uygulama
    
    lam=0 → TD(0)
    lam=1 → Monte Carlo eşdeğeri
    """
    V      = np.zeros(mrp.n_states)
    errors = []
    
    for ep in range(n_episodes):
        z     = np.zeros(mrp.n_states)   # Uygunluk izleri
        state = mrp.states[len(mrp.states) // 2]
        
        for _ in range(500):
            next_state, reward = mrp.step(state)
            i = mrp.state_idx[state]
            j = mrp.state_idx[next_state]
            
            # TD hatası
            delta = reward + mrp.gamma * V[j] - V[i]
            
            # İz güncelleme (biriktirme)
            z    *= mrp.gamma * lam
            z[i] += 1.0
            
            # Tüm durumları güncelle
            V += alpha * delta * z
            
            if next_state in [0, mrp.n_states - 1]:
                z[:] = 0  # Terminal: izleri sıfırla
                state = mrp.states[len(mrp.states) // 2]
            else:
                state = next_state
        
        if ep % 30 == 0:
            rmse = np.sqrt(np.mean((V - V_true) ** 2))
            errors.append(rmse)
    
    return V, errors

# Farklı λ değerleri
lambdas = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]
colors  = plt.cm.plasma(np.linspace(0, 0.9, len(lambdas)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

final_rmse = []
for lam, col in zip(lambdas, colors):
    V_lam, errs_lam = td_lambda(mrp, lam=lam, alpha=0.05, n_episodes=2_000)
    ep_ticks = np.arange(len(errs_lam)) * 30
    axes[0].plot(ep_ticks, errs_lam, color=col, linewidth=2, label=f'λ={lam}')
    final_rmse.append(errs_lam[-1])

axes[0].set_xlabel('Epizot')
axes[0].set_ylabel('RMSE')
axes[0].set_title('TD(λ): Farklı λ Değerleri (α=0.05)', fontweight='bold')
axes[0].legend(fontsize=9)

# Final RMSE vs λ
axes[1].plot(lambdas, final_rmse, 'ko-', markersize=8, linewidth=2)
axes[1].fill_between(lambdas, final_rmse, alpha=0.2)
axes[1].set_xlabel('λ değeri')
axes[1].set_ylabel('Final RMSE (2000 epizot sonrası)')
axes[1].set_title('TD(λ): λ Seçiminin Etkisi', fontweight='bold')
axes[1].axvline(x=lambdas[np.argmin(final_rmse)], color='red', linestyle='--',
                label=f'En iyi λ = {lambdas[np.argmin(final_rmse)]}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_td_lambda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"En iyi λ = {lambdas[np.argmin(final_rmse)]} (RMSE = {min(final_rmse):.5f})")

## 2.5 Büyük Durum Uzayları: Doğrusal Fonksiyon Yaklaşımı

Durum uzayı büyüdüğünde tabular temsil yetersiz kalır. **Fonksiyon yaklaşımı** kullanırız:

$$\hat{V}(x; \theta) = \theta^\top \phi(x)$$

Burada $\phi(x) \in \mathbb{R}^d$ özellik vektörüdür ve $d \ll |\mathcal{X}|$.

### TD(λ) ile Fonksiyon Yaklaşımı (Szepesvári, s.22)

$$\theta_{t+1} = \theta_t + \alpha_t \delta_{t+1} z_{t+1}$$

Uygunluk izi:
$$z_{t+1} = \gamma \lambda z_t + \phi(X_t)$$

TD hatası:
$$\delta_{t+1} = R_{t+1} + \gamma \hat{V}(X_{t+1}; \theta_t) - \hat{V}(X_t; \theta_t)$$


In [ ]:
def create_mountaincar_features(n_tiles=8, n_tilings=4):
    """
    Mountain Car benzeri sürekli durum uzayı için tile coding özellikleri.
    Durum: (konum, hız)
    """
    pos_range   = (-1.2, 0.6)
    vel_range   = (-0.07, 0.07)
    n_features  = n_tiles * n_tiles * n_tilings
    
    def phi(state):
        """Tile coding özellik vektörü"""
        pos, vel = state
        feat = np.zeros(n_features)
        for tiling in range(n_tilings):
            # Her katman için hafif kaydırma
            offset_p = tiling * (pos_range[1] - pos_range[0]) / (n_tiles * n_tilings)
            offset_v = tiling * (vel_range[1] - vel_range[0]) / (n_tiles * n_tilings)
            
            p_idx = int((pos - pos_range[0] + offset_p) / 
                        (pos_range[1] - pos_range[0]) * n_tiles) % n_tiles
            v_idx = int((vel - vel_range[0] + offset_v) / 
                        (vel_range[1] - vel_range[0]) * n_tiles) % n_tiles
            
            feat_idx = tiling * n_tiles * n_tiles + p_idx * n_tiles + v_idx
            feat[feat_idx] = 1.0
        return feat
    
    return phi, n_features

# Yapay sürekli ortam simülasyonu
def linear_td0_continuous(n_states_sim=200, n_features=50, n_steps=20_000, alpha=0.01, gamma=0.95):
    """
    Sürekli durum uzayında doğrusal TD(0).
    Szepesvári s.22 - Algoritma 4'ün basit versiyonu.
    """
    # Rastgele özellik matrisi (her durum için sabit)
    np.random.seed(42)
    Phi = np.random.randn(n_states_sim, n_features)
    Phi /= np.linalg.norm(Phi, axis=1, keepdims=True)  # Normalize
    
    # Gerçek değerler (yapay)
    true_weights = np.random.randn(n_features)
    V_true_cont  = Phi @ true_weights
    
    # Geçiş matrisi (rastgele yürüyüş)
    P = np.zeros((n_states_sim, n_states_sim))
    for i in range(n_states_sim):
        neighbors = [max(0, i-3), max(0, i-1), i, 
                     min(n_states_sim-1, i+1), min(n_states_sim-1, i+3)]
        for nb in neighbors:
            P[i, nb] += 1.0
        P[i] /= P[i].sum()
    
    # Ödül fonksiyonu
    R_fn = 0.1 * np.random.randn(n_states_sim)
    
    # TD(0) öğrenimi
    theta  = np.zeros(n_features)
    errors = []
    state  = n_states_sim // 2
    
    for t in range(n_steps):
        probs      = P[state]
        next_state = np.random.choice(n_states_sim, p=probs)
        reward     = R_fn[state]
        
        # TD güncelleme: θ ← θ + α·δ·ϕ(s)
        phi_s      = Phi[state]
        phi_ns     = Phi[next_state]
        V_s        = theta @ phi_s
        V_ns       = theta @ phi_ns
        td_error   = reward + gamma * V_ns - V_s
        theta     += alpha * td_error * phi_s
        
        state = next_state
        
        if t % 200 == 0:
            V_est = Phi @ theta
            rmse  = np.sqrt(np.mean((V_est - V_true_cont) ** 2))
            errors.append(rmse)
    
    V_est = Phi @ theta
    return V_est, V_true_cont, errors, theta

V_est, V_true_cont, errors_fa, theta = linear_td0_continuous()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sol: Gerçek vs tahmin
x_axis = np.arange(len(V_true_cont))
axes[0].plot(x_axis, V_true_cont, 'k-',  linewidth=2, label='Gerçek V', alpha=0.7)
axes[0].plot(x_axis, V_est,       'r--', linewidth=2, label='TD(0) Tahmini (Doğrusal FA)')
axes[0].set_xlabel('Durum indeksi')
axes[0].set_ylabel('Değer')
axes[0].set_title('Doğrusal Fonksiyon Yaklaşımı ile TD(0)', fontweight='bold')
axes[0].legend()

# Sağ: Yakınsama
step_ax = np.arange(len(errors_fa)) * 200
axes[1].semilogy(step_ax, errors_fa, 'purple', linewidth=2)
axes[1].set_xlabel('Adım Sayısı')
axes[1].set_ylabel('RMSE (log ölçek)')
axes[1].set_title('Doğrusal TD(0) Yakınsama Eğrisi', fontweight='bold')
axes[1].fill_between(step_ax, errors_fa, alpha=0.2, color='purple')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_func_approx.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final RMSE: {errors_fa[-1]:.5f}")

## 2.6 Least-Squares TD (LSTD)

Gradient TD öğrenmesi yerine **en küçük kareler** yöntemini kullanarak daha verimli bir çözüm bulabiliriz.

### LSTD(0) — Algoritma 7 (Szepesvári, s.29)

LSTD doğrudan şu lineer sistemi çözer:

$$A \theta = b$$

$$A = \sum_{t} \phi(X_t)(\phi(X_t) - \gamma \phi(X_{t+1}))^\top, \quad b = \sum_t \phi(X_t) R_{t+1}$$

**Avantaj:** İteratif güncelleme gerektirmez, tek seferde çözülür.  
**Dezavantaj:** $O(d^2)$ bellek, $O(d^3)$ hesaplama maliyeti.


In [ ]:
def lstd(mrp_phi, Phi, n_steps=5_000, gamma=0.95):
    """
    LSTD(0) — Least-Squares Temporal Difference
    Szepesvári Algoritma 7 (s.29) basit versiyonu.
    
    mrp_phi : (state, next_state, reward) üçlüleri üretebilen nesne
    Phi     : (n_states x d) özellik matrisi
    """
    n_states, d = Phi.shape
    A = np.zeros((d, d))
    b = np.zeros(d)
    
    state = n_states // 2
    
    # Geçiş matrisi (basit random walk)
    P = np.zeros((n_states, n_states))
    for i in range(n_states):
        neighbors = [max(0, i-1), min(n_states-1, i+1)]
        for nb in neighbors:
            P[i, nb] += 0.5
    
    R_fn = np.zeros(n_states)
    R_fn[-2] = 1.0  # Sona yakın ödül
    
    for t in range(n_steps):
        probs      = P[state]
        next_state = np.random.choice(n_states, p=probs)
        reward     = R_fn[state]
        
        phi_s  = Phi[state]
        phi_ns = Phi[next_state]
        
        # A ve b birikimi
        A += np.outer(phi_s, phi_s - gamma * phi_ns)
        b += phi_s * reward
        
        state = next_state
    
    # Doğrusal sistemi çöz: A theta = b
    reg = 1e-6 * np.eye(d)   # Regularizasyon
    theta_lstd = np.linalg.solve(A + reg, b)
    
    return theta_lstd, A, b

# Karşılaştırma: TD(0) vs LSTD vs Gerçek
np.random.seed(0)
n_s, d = 50, 10
Phi_test = np.random.randn(n_s, d)
Phi_test /= np.linalg.norm(Phi_test, axis=1, keepdims=True)

theta_true = np.random.randn(d)
V_true_test = Phi_test @ theta_true

# LSTD
theta_lstd, A_mat, b_vec = lstd(None, Phi_test, n_steps=10_000)
V_lstd = Phi_test @ theta_lstd

# TD(0) doğrusal
_, V_td_fa, errs_cmp, theta_td = linear_td0_continuous(
    n_states_sim=n_s, n_features=d, n_steps=10_000)

# Özellik matrisini aynı tut
V_td_fa2 = Phi_test @ theta_td[:d] if len(theta_td) >= d else V_td_fa[:n_s]

fig, ax = plt.subplots(figsize=(12, 5))
x_ax = np.arange(n_s)
ax.plot(x_ax, V_true_test, 'k-',  linewidth=3, label='Gerçek V', zorder=5)
ax.plot(x_ax, V_lstd,      'r--', linewidth=2, label=f'LSTD (RMSE={np.sqrt(np.mean((V_lstd-V_true_test)**2)):.4f})')

ax.set_xlabel('Durum indeksi')
ax.set_ylabel('Değer')
ax.set_title('LSTD(0): Least-Squares Temporal Difference Karşılaştırması', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_lstd.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"LSTD RMSE: {np.sqrt(np.mean((V_lstd - V_true_test)**2)):.5f}")

## 2.7 Kapsamlı Karşılaştırma

Her yöntemin avantaj ve dezavantajlarını görselleştirelim:


In [ ]:
# ── Kapsamlı algoritma karşılaştırması ───────────────────────────
mrp_cmp = create_random_walk(n_states=7, gamma=0.95)
V_true_cmp = mrp_cmp.true_value()

results = {}
for alpha_val in [0.02, 0.05, 0.1]:
    # TD(0)
    V_td0, _ = td0(mrp_cmp, n_steps=20_000, alpha=alpha_val)
    # MC
    V_mc_a, _ = every_visit_mc(mrp_cmp, n_episodes=2_000, alpha=alpha_val)
    # TD(0.5)
    V_td5, _ = td_lambda(mrp_cmp, lam=0.5, alpha=alpha_val, n_episodes=2_000)
    
    results[alpha_val] = {
        'TD(0)':    np.sqrt(np.mean((V_td0 - V_true_cmp)**2)),
        'MC':       np.sqrt(np.mean((V_mc_a - V_true_cmp)**2)),
        'TD(0.5)':  np.sqrt(np.mean((V_td5 - V_true_cmp)**2)),
    }

fig, ax = plt.subplots(figsize=(11, 5))
methods = ['TD(0)', 'MC', 'TD(0.5)']
alphas_plot = [0.02, 0.05, 0.1]
x = np.arange(len(methods))
width = 0.25
colors_bar = ['steelblue', 'darkorange', 'green']

for i, (alpha_val, col) in enumerate(zip(alphas_plot, colors_bar)):
    rmses = [results[alpha_val][m] for m in methods]
    ax.bar(x + i*width, rmses, width, label=f'α={alpha_val}', color=col, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(methods, fontsize=12)
ax.set_ylabel('Final RMSE')
ax.set_title('Değer Tahmin Algoritmaları Karşılaştırması\n(20K adım / 2K epizot sonrası)', fontweight='bold')
ax.legend()

# En iyi kombinasyonu işaretle
best_method = min(methods, key=lambda m: results[0.05][m])
print(f"En iyi yöntem (α=0.05): {best_method}")

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n─── Sonuç Tablosu ───")
print(f"{'Yöntem':<15} {'α=0.02':>10} {'α=0.05':>10} {'α=0.10':>10}")
print("─" * 50)
for m in methods:
    row = f"{m:<15}"
    for a in alphas_plot:
        row += f"{results[a][m]:>10.5f}"
    print(row)

## 2.8 Özet

| Algoritma | Önyükleme | Terminal Gerekli | Yakınsama | Varyans |
|-----------|-----------|-----------------|-----------|---------|
| **TD(0)**  | ✅ Evet | ❌ Hayır | Kesin (RM koşulları) | Düşük |
| **Monte Carlo** | ❌ Hayır | ✅ Evet | Kesin (RM koşulları) | Yüksek |
| **TD(λ)** | ✅ Kısmen | ✅/❌ λ'ya bağlı | Kesin (RM koşulları) | Orta |
| **LSTD(0)** | ✅ Evet | ❌ Hayır | Tek adımda | Çok düşük |

### Ne zaman hangisi?
- **TD(0):** Online öğrenmede, durum uzayı makul boyuttaysa.
- **Monte Carlo:** Epizodik problem, bootstrapping bias'ından kaçınmak için.
- **TD(λ):** İkisini dengelemek için; λ hiperparametre olarak ayarlanır.
- **LSTD:** Verinin toplanıp sonra analiz edildiği batch senaryolarında.
